In [1]:
# Author: Arthur Prigent
# Email: arthur.prigent@univ-brest.fr

In [2]:
import matplotlib.patches as mpatches
from scipy.stats import pearsonr
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy
import matplotlib
import scipy
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import matplotlib.colors as mcolors

from datetime import datetime, timedelta
import cartopy.feature as cfeature
import numpy as np
import xarray as xr
import glob
import matplotlib.pyplot as plt
import netCDF4
import string
from scipy.spatial import cKDTree
from datetime import timedelta
import gsw

### PSAL

In [3]:
ds_all_psal = xr.open_dataset('/data0/user/aprigent/PROCESSED/level_2/profiles_QC_PSAL_new2.nc',decode_times=False)

### TEMP

In [4]:
ds_all_temp = xr.open_dataset('/data0/user/aprigent/PROCESSED/level_2/profiles_QC_TEMP_new2.nc',decode_times=False)

In [5]:
def build_key(ds):
    # safer precision: avoid too coarse rounding
    time   = np.char.mod('%.5f', ds.time.values)
    lat    = np.char.mod('%.5f', ds.latitude.values)
    lon    = np.char.mod('%.5f', ds.longitude.values)
    source = ds.source.values.astype(str)

    key = np.char.add(time, "_")
    key = np.char.add(key, lat)
    key = np.char.add(key, "_")
    key = np.char.add(key, lon)
    key = np.char.add(key, "_")
    key = np.char.add(key, source)

    return key

def drop_duplicate_profiles(ds):
    key = build_key(ds)
    _, unique_idx = np.unique(key, return_index=True)
    n_before = ds.profile.size
    ds_deduped = ds.isel(profile=unique_idx)
    n_dropped = n_before - ds_deduped.profile.size
    print(f"Dropped {n_dropped} duplicate profiles, {ds_deduped.profile.size} remaining")
    return ds_deduped

ds_temp_test = drop_duplicate_profiles(ds_all_temp)
ds_psal_test = drop_duplicate_profiles(ds_all_psal)

Dropped 0 duplicate profiles, 361904 remaining
Dropped 0 duplicate profiles, 329631 remaining


In [6]:
# --- Find common profiles ---
key_temp = build_key(ds_all_temp)
key_psal = build_key(ds_all_psal)

common_keys = np.intersect1d(key_temp, key_psal)

ds_temp_common = ds_all_temp.isel(profile=np.isin(key_temp, common_keys))
ds_psal_common = ds_all_psal.isel(profile=np.isin(key_psal, common_keys))

print(f"Common profiles: {ds_temp_common.profile.size}")

Common profiles: 320609


In [7]:
ds_temp_common["source"] = ds_temp_common["source"].astype(str)
ds_psal_common["source"] = ds_psal_common["source"].astype(str)

In [8]:
ds_temp_common_qc_UDASH = ds_temp_common.where(ds_temp_common["source"]=='UDASH',drop=True)
ds_temp_common_qc_ITP = ds_temp_common.where(ds_temp_common["source"]=='ITP',drop=True)
ds_temp_common_qc_ARGO = ds_temp_common.where(ds_temp_common["source"]=='ARGO',drop=True)
ds_temp_common_qc_NABOS = ds_temp_common.where(ds_temp_common["source"]=='NABOS_ctd',drop=True)
ds_temp_common_qc_BGEP = ds_temp_common.where(ds_temp_common["source"]=='BGEP_ctd',drop=True)
ds_temp_common_qc_MEOP = ds_temp_common.where(ds_temp_common["source"]=='MEOP',drop=True)
ds_temp_common_qc_ICES = ds_temp_common.where(ds_temp_common["source"]=='ICES',drop=True)
ds_temp_common_qc_CORA = ds_temp_common.where(ds_temp_common["source"]=='CORA',drop=True)
ds_temp_common_qc_WOD = ds_temp_common.where(ds_temp_common["source"]=='WOD',drop=True)


ds_psal_common_qc_UDASH = ds_psal_common.where(ds_psal_common["source"]=='UDASH',drop=True)
ds_psal_common_qc_ITP = ds_psal_common.where(ds_psal_common["source"]=='ITP',drop=True)
ds_psal_common_qc_ARGO = ds_psal_common.where(ds_psal_common["source"]=='ARGO',drop=True)
ds_psal_common_qc_NABOS = ds_psal_common.where(ds_psal_common["source"]=='NABOS_ctd',drop=True)
ds_psal_common_qc_BGEP = ds_psal_common.where(ds_psal_common["source"]=='BGEP_ctd',drop=True)
ds_psal_common_qc_MEOP = ds_psal_common.where(ds_psal_common["source"]=='MEOP',drop=True)
ds_psal_common_qc_ICES = ds_psal_common.where(ds_psal_common["source"]=='ICES',drop=True)
ds_psal_common_qc_CORA = ds_psal_common.where(ds_psal_common["source"]=='CORA',drop=True)
ds_psal_common_qc_WOD = ds_psal_common.where(ds_psal_common["source"]=='WOD',drop=True)

In [9]:
ds_temp_common_qc_UDASH.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_UDASH_ISAS_TEMP_common.nc')
ds_temp_common_qc_ITP.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_ITP_ISAS_TEMP_common.nc')
ds_temp_common_qc_ARGO.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_ARGO_ISAS_TEMP_common.nc')
ds_temp_common_qc_NABOS.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_NABOS_ISAS_TEMP_common.nc')
ds_temp_common_qc_BGEP.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_BGEP_ISAS_TEMP_common.nc')
ds_temp_common_qc_MEOP.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_MEOP_ISAS_TEMP_common.nc')
ds_temp_common_qc_ICES.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_ICES_ISAS_TEMP_common.nc')
ds_temp_common_qc_CORA.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_CORA_ISAS_TEMP_common.nc')
ds_temp_common_qc_WOD.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_WOD_ISAS_TEMP_common.nc')


ds_psal_common_qc_UDASH.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_UDASH_ISAS_PSAL_common.nc')
ds_psal_common_qc_ITP.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_ITP_ISAS_PSAL_common.nc')
ds_psal_common_qc_ARGO.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_ARGO_ISAS_PSAL_common.nc')
ds_psal_common_qc_NABOS.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_NABOS_ISAS_PSAL_common.nc')
ds_psal_common_qc_BGEP.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_BGEP_ISAS_PSAL_common.nc')
ds_psal_common_qc_MEOP.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_MEOP_ISAS_PSAL_common.nc')
ds_psal_common_qc_ICES.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_ICES_ISAS_PSAL_common.nc')
ds_psal_common_qc_CORA.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_CORA_ISAS_PSAL_common.nc')
ds_psal_common_qc_WOD.to_netcdf('/data0/user/aprigent/PROCESSED/level_3/arctic_qc_WOD_ISAS_PSAL_common.nc')